# 📈 StokSignal — прогноз акций MOEX + Telegram-бот

**Команда:** Бурая А. · Бабаев Э. · Погосян А. · Святская П. · Дзугаева М.

Что делает этот ноутбук:
1. Загружает данные с Московской биржи (SBER, GAZP, LKOH)
2. Обучает модель машинного обучения (GBR + LSTM)
3. Строит графики как в презентации
4. Генерирует торговые сигналы BUY / SELL / HOLD
5. Сохраняет модель в файл `.pkl`
6. Telegram-бот каждый день в 9:00 пишет что делать по каждой акции

---

## ⚙️ Шаг 1 — Установка

In [ ]:
!pip install aiomoex nest_asyncio --quiet
print('✅ Готово')

---
## 📥 Шаг 2 — Загрузка данных с Московской биржи

Данные берём напрямую с ISS MOEX — бесплатно и официально.
- **SBER** и **LKOH** — с 2019 года
- **GAZP** — с августа 2022 (после сплита акций 1:40 в июле 2022)

In [ ]:
import asyncio, aiohttp, aiomoex
import pandas as pd, numpy as np, warnings, time
warnings.filterwarnings('ignore')
from datetime import date

try:
    import nest_asyncio; nest_asyncio.apply()
except: pass

TICKERS    = ['SBER', 'GAZP', 'LKOH']
DATE_END   = date.today().isoformat()
TICKER_START = {'SBER': '2019-01-01', 'GAZP': '2022-08-01', 'LKOH': '2019-01-01'}
COLUMNS    = ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'VOLUME']

async def load_ticker(session, ticker):
    data = await aiomoex.get_board_history(
        session, ticker, start=TICKER_START[ticker], end=DATE_END,
        columns=['TRADEDATE']+COLUMNS)
    df = pd.DataFrame(data)
    df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'])
    return df.set_index('TRADEDATE').sort_index().astype(float)[COLUMNS]

async def load_all():
    result = {}
    async with aiohttp.ClientSession() as session:
        for t in TICKERS:
            try:
                result[t] = await load_ticker(session, t)
                print(f"  ✓ {t}: {len(result[t])} дней  "
                      f"({result[t].index[0].date()} → {result[t].index[-1].date()})  "
                      f"цена: {result[t]['CLOSE'].min():.1f}–{result[t]['CLOSE'].max():.1f} руб.")
            except Exception as e:
                print(f"  ✗ {t}: {e}")
    return result

print(f"Загружаем данные до {DATE_END}...")
t0 = time.time()
raw_data = asyncio.run(load_all())
print(f"\n✅ Загружено за {time.time()-t0:.1f} сек.")

---
## 🧹 Шаг 3 — Очистка данных

- Пропуски → `ffill`
- Выбросы → IQR, скользящее окно 20 дней
- Объём → логарифмируем
- Разбивка → train 80% / test 20% строго по времени

In [ ]:
def remove_outliers(s, w=20):
    s = s.copy()
    q1 = s.rolling(w, min_periods=5).quantile(0.25)
    q3 = s.rolling(w, min_periods=5).quantile(0.75)
    med = s.rolling(w, min_periods=5).median()
    s[(s < q1-1.5*(q3-q1)) | (s > q3+1.5*(q3-q1))] = med
    return s

def clean(df):
    d = df.copy().ffill().dropna()
    d['CLOSE'] = remove_outliers(d['CLOSE'])
    d['VOLUME'] = remove_outliers(d['VOLUME'])
    d['VOLUME_LOG'] = np.log1p(d['VOLUME'])
    return d

SPLIT = 0.80
clean_data = {t: clean(df) for t, df in raw_data.items()}
splits = {}
for t, df in clean_data.items():
    n = len(df); idx = int(n*SPLIT)
    splits[t] = {'df': df, 'train': df.iloc[:idx], 'test': df.iloc[idx:]}

print("Разбивка train / test:")
print("-"*65)
for t, s in splits.items():
    tr, te = s['train'], s['test']
    print(f"  {t}: train {len(tr)} ({tr.index[0].date()}–{tr.index[-1].date()}) | "
          f"test {len(te)} ({te.index[0].date()}–{te.index[-1].date()})")

---
## 🔧 Шаг 4 — Признаки для модели (11 штук)

In [ ]:
def rsi(s, p=14):
    d = s.diff()
    g = d.clip(lower=0).rolling(p).mean()
    l = (-d.clip(upper=0)).rolling(p).mean()
    return 100 - 100/(1 + g/l.replace(0, np.nan))

FEATURES = ['Lag_1','Lag_5','Lag_10','MA_5','MA_20','STD_10',
            'RSI_14','Return_1d','DayOfWeek','Month','VOLUME_LOG']
TARGET = 'CLOSE'

def build_features(df):
    d = df.copy()
    d['Lag_1']     = d['CLOSE'].shift(1)
    d['Lag_5']     = d['CLOSE'].shift(5)
    d['Lag_10']    = d['CLOSE'].shift(10)
    d['MA_5']      = d['CLOSE'].rolling(5).mean()
    d['MA_20']     = d['CLOSE'].rolling(20).mean()
    d['STD_10']    = d['CLOSE'].rolling(10).std()
    d['RSI_14']    = rsi(d['CLOSE'])
    d['Return_1d'] = d['CLOSE'].pct_change()
    d['DayOfWeek'] = d.index.dayofweek
    d['Month']     = d.index.month
    return d.dropna(subset=FEATURES+[TARGET])

feat = {t: build_features(s['df']) for t, s in splits.items()}
for t, df in feat.items():
    print(f"  {t}: {len(df)} строк × {len(FEATURES)} признаков")

---
## 🤖 Шаг 5 — Обучение моделей

- **Naive** — бенчмарк «завтра = сегодня»
- **Linear Regression** — baseline
- **Random Forest** — ансамбль деревьев
- **GBR ★** — финальная модель
- **LSTM** — нейросеть для временных рядов

In [ ]:
from sklearn.linear_model   import LinearRegression
from sklearn.ensemble       import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics        import mean_absolute_error, r2_score
from sklearn.preprocessing  import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models    import Sequential
from tensorflow.keras.layers    import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
tf.get_logger().setLevel('ERROR')

def mape_fn(yt, yp):
    yt, yp = np.array(yt), np.array(yp)
    m = yt != 0
    return np.mean(np.abs((yt[m]-yp[m])/yt[m]))*100

LSTM_WIN = 20
results = {}; trained = {}; preds = {}

for ticker, df in feat.items():
    X = df[FEATURES].values; y = df[TARGET].values
    n = len(X); sp = int(n*SPLIT)
    X_tr, X_te, y_tr, y_te = X[:sp], X[sp:], y[:sp], y[sp:]
    results[ticker] = {}; trained[ticker] = {}
    preds[ticker]   = {'y_test': df[TARGET].iloc[sp:]}

    # Naive
    yn = df['Lag_1'].values[sp:]; ml = min(len(y_te), len(yn))
    results[ticker]['Naive'] = {'MAE': round(mean_absolute_error(y_te[:ml],yn[:ml]),2),
                                 'MAPE': round(mape_fn(y_te[:ml],yn[:ml]),2), 'R2': '—'}

    # Linear
    lr = LinearRegression().fit(X_tr,y_tr); yp = lr.predict(X_te)
    results[ticker]['Linear'] = {'MAE':round(mean_absolute_error(y_te,yp),2),
                                  'MAPE':round(mape_fn(y_te,yp),2),'R2':round(r2_score(y_te,yp),4)}
    trained[ticker]['Linear'] = lr

    # RF
    rf = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
    rf.fit(X_tr,y_tr); yp = rf.predict(X_te)
    results[ticker]['Random Forest'] = {'MAE':round(mean_absolute_error(y_te,yp),2),
                                         'MAPE':round(mape_fn(y_te,yp),2),'R2':round(r2_score(y_te,yp),4)}
    trained[ticker]['Random Forest'] = rf

    # GBR
    gbr = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=42)
    gbr.fit(X_tr,y_tr); yp = gbr.predict(X_te)
    results[ticker]['GBR ★'] = {'MAE':round(mean_absolute_error(y_te,yp),2),
                                    'MAPE':round(mape_fn(y_te,yp),2),'R2':round(r2_score(y_te,yp),4)}
    trained[ticker]['GBR ★'] = gbr
    preds[ticker]['GBR ★'] = yp

    # LSTM
    sx = MinMaxScaler(); sy = MinMaxScaler()
    Xs = sx.fit_transform(X); ys = sy.fit_transform(y.reshape(-1,1)).ravel()
    def mk_seq(Xd,yd,w):
        xs_,ys_ = [],[]
        for i in range(len(Xd)-w): xs_.append(Xd[i:i+w]); ys_.append(yd[i+w])
        return np.array(xs_), np.array(ys_)
    Xs_,ys_ = mk_seq(Xs,ys,LSTM_WIN)
    # split по индексам последовательностей, а не по исходным данным
    # последовательность i соответствует исходному индексу i+LSTM_WIN
    # поэтому тестовые последовательности: те где i+LSTM_WIN >= sp
    ssp = sp - LSTM_WIN  # первый тестовый индекс в Xs_
    ssp = max(0, ssp)
    lstm = Sequential([LSTM(64,input_shape=(LSTM_WIN,len(FEATURES)),return_sequences=True),
                       Dropout(0.2),LSTM(32),Dropout(0.2),Dense(1)])
    lstm.compile('adam','mse')
    lstm.fit(Xs_[:ssp],ys_[:ssp],epochs=50,batch_size=32,validation_split=0.1,
             callbacks=[EarlyStopping(patience=5,restore_best_weights=True)],verbose=0)
    yp_s  = lstm.predict(Xs_[ssp:],verbose=0).ravel()
    yp_l  = sy.inverse_transform(yp_s.reshape(-1,1)).ravel()
    y_tl  = ys_[ssp:]  # целевые значения — тоже из последовательностей
    y_tl  = sy.inverse_transform(y_tl.reshape(-1,1)).ravel()
    # обрезаем до одинаковой длины на случай краевых эффектов
    min_len = min(len(y_tl), len(yp_l))
    y_tl, yp_l = y_tl[:min_len], yp_l[:min_len]
    results[ticker]['LSTM'] = {'MAE':round(mean_absolute_error(y_tl,yp_l),2),
                                'MAPE':round(mape_fn(y_tl,yp_l),2),'R2':round(r2_score(y_tl,yp_l),4)}
    trained[ticker]['LSTM'] = {'model':lstm,'sx':sx,'sy':sy}
    preds[ticker]['LSTM'] = yp_l
    print(f"  {ticker} ✓  GBR: MAPE={results[ticker]['GBR ★']['MAPE']}%  R²={results[ticker]['GBR ★']['R2']}")

print()
print("="*68)
print(f"{'Тикер':<8} {'Модель':<16} {'MAE':>8} {'MAPE':>8} {'R²':>8}")
print("="*68)
for t in TICKERS:
    for name, m in results[t].items():
        mark = ' ←' if '★' in name else ''
        print(f"{t:<8} {name:<16} {m['MAE']:>8.2f} {m['MAPE']:>7.2f}% {str(m['R2']):>8}{mark}")
    print("-"*68)

---
## 📊 Шаг 6 — Графики

1. Прогноз GBR vs Факт
2. Важность признаков (SBER)
3. Сравнение MAPE всех моделей

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

BG='#0A0F1E'; CARD='#141E3C'; CYAN='#00E5FF'
AMBER='#FFB300'; WHITE='#FFFFFF'; GRAY='#8090A0'

# ── График 1: Прогноз vs Факт ─────────────────────────────────────
fig, axes = plt.subplots(1,3,figsize=(18,5.5))
fig.patch.set_facecolor(BG)
fig.suptitle('StokSignal — Прогноз GBR vs Факт (тестовая выборка 20%)',
             fontsize=14, color=WHITE, fontweight='bold', y=1.01)
for ax, ticker in zip(axes, TICKERS):
    ax.set_facecolor(CARD)
    yt = preds[ticker]['y_test'].values
    yp = preds[ticker]['GBR ★']
    n  = min(len(yt), len(yp))
    ax.plot(range(n), yt[:n], color=WHITE, lw=1.6, alpha=0.9, label='Факт')
    ax.plot(range(n), yp[:n], color=CYAN,  lw=1.3, alpha=0.85, ls='--', label='Прогноз GBR')
    ax.fill_between(range(n), yt[:n], yp[:n], color=CYAN, alpha=0.07)
    m = results[ticker]['GBR ★']
    ax.set_title(f"{ticker}  MAPE={m['MAPE']}%  R²={m['R2']}",
                 color=CYAN, fontsize=12, fontweight='bold', pad=8)
    ax.set_xlabel('Торговые дни (тест)', color=GRAY, fontsize=9)
    ax.set_ylabel('Цена, руб.', color=GRAY, fontsize=9)
    ax.tick_params(colors=GRAY, labelsize=8)
    for sp2 in ax.spines.values(): sp2.set_edgecolor('#2A3A5A')
    ax.legend(fontsize=9, facecolor='#1A2744', labelcolor=WHITE, edgecolor='#2A3A5A')
    ax.grid(color='#1A2744', alpha=0.6, lw=0.5)
plt.tight_layout(pad=1.5); plt.show()

# ── График 2: Важность признаков ─────────────────────────────────
imps = trained['SBER']['GBR ★'].feature_importances_
idx2 = np.argsort(imps)
vals = imps[idx2]*100; labs = [FEATURES[i] for i in idx2]
cols = [AMBER if v>8 else CYAN if v>2 else '#3A5A7A' for v in vals]
fig, ax = plt.subplots(figsize=(10,6))
fig.patch.set_facecolor(BG); ax.set_facecolor(CARD)
bars = ax.barh(labs, vals, color=cols, edgecolor='none', height=0.65)
for bar,val in zip(bars,vals):
    if val>0.3:
        ax.text(val+0.3, bar.get_y()+bar.get_height()/2,
                f'{val:.1f}%', va='center', color=WHITE, fontsize=10, fontweight='bold')
ax.set_title('Важность признаков GBR — SBER', color=CYAN, fontsize=13, fontweight='bold', pad=10)
ax.set_xlabel('Вклад, %', color=GRAY, fontsize=11)
ax.tick_params(colors=WHITE, labelsize=10)
ax.set_xlim(0, vals.max()*1.22)
for sp2 in ax.spines.values(): sp2.set_edgecolor('#2A3A5A')
ax.grid(axis='x', color='#1A2744', alpha=0.6, lw=0.5)
ax.legend(handles=[mpatches.Patch(color=AMBER,label='Ключевые (>8%)'),
                   mpatches.Patch(color=CYAN, label='Значимые (>2%)'),
                   mpatches.Patch(color='#3A5A7A',label='Второстепенные')],
          fontsize=9, facecolor='#1A2744', labelcolor=WHITE, edgecolor='#2A3A5A', loc='lower right')
plt.tight_layout(); plt.show()

# ── График 3: Сравнение MAPE ──────────────────────────────────────
mnames = ['Naive
(бенчмарк)','Linear
Regression','Random
Forest','GBR ★']
mkeys  = ['Naive','Linear','Random Forest','GBR ★']
def get_m(t, k): return results[t].get(k, {}).get('MAPE', 0)
x = np.arange(len(mnames)); w = 0.25
fig, ax = plt.subplots(figsize=(12,6))
fig.patch.set_facecolor(BG); ax.set_facecolor(CARD)
b1=ax.bar(x-w,[get_m(t,k) for t,k in [('SBER',mk) for mk in mkeys]],w,label='SBER',color='#4FC3F7',edgecolor='none')
b2=ax.bar(x,  [get_m(t,k) for t,k in [('GAZP',mk) for mk in mkeys]],w,label='GAZP',color='#FFB300',edgecolor='none')
b3=ax.bar(x+w,[get_m(t,k) for t,k in [('LKOH',mk) for mk in mkeys]],w,label='LKOH',color='#81C784',edgecolor='none')
ax.axhline(2.0, color='#FF5252', lw=2, ls='--')
ax.text(len(mnames)-0.55, 2.18, 'Цель < 2%', color='#FF5252', fontsize=10, fontweight='bold')
for bars_grp in [b1,b2,b3]:
    last = bars_grp[-1]
    ax.text(last.get_x()+last.get_width()/2, last.get_height()+0.1,
            f'{last.get_height():.2f}%', ha='center', color=WHITE, fontsize=10, fontweight='bold')
ax.set_title('MAPE по моделям — чем меньше тем лучше', color=CYAN, fontsize=13, fontweight='bold', pad=10)
ax.set_ylabel('MAPE, %', color=GRAY, fontsize=11)
ax.set_xticks(x); ax.set_xticklabels(mnames, color=WHITE, fontsize=11)
ax.tick_params(colors=GRAY)
max_v = max(get_m(t,k) for t in TICKERS for k in mkeys)
ax.set_ylim(0, max_v*1.18)
for sp2 in ax.spines.values(): sp2.set_edgecolor('#2A3A5A')
ax.grid(axis='y', color='#1A2744', alpha=0.6, lw=0.5)
ax.legend(fontsize=10, facecolor='#1A2744', labelcolor=WHITE, edgecolor='#2A3A5A')
plt.tight_layout(); plt.show()
print("\nGBR лучший по всем тикерам ✅")

---
## 🚦 Шаг 7 — Торговые сигналы

- прогноз выше текущей цены на **>3%** → 🟢 КУПИТЬ
- прогноз ниже текущей цены на **>3%** → 🔴 ПРОДАТЬ
- иначе → ⚪ ДЕРЖАТЬ

In [ ]:
THRESHOLD = 0.03

def get_signals(ticker):
    df  = feat[ticker]
    n   = len(df); sp = int(n*SPLIT)
    yp  = trained[ticker]['GBR ★'].predict(df[FEATURES].iloc[sp:].values)
    out = df[[TARGET]].iloc[sp:].copy().rename(columns={TARGET:'Цена_факт'})
    out['Прогноз']    = yp.round(2)
    out['Изменение_%']= ((out['Прогноз']-out['Цена_факт'])/out['Цена_факт']*100).round(2)
    out['Сигнал']     = out['Изменение_%'].apply(
        lambda d: '⚠️ аномалия' if abs(d)>50
             else '🟢 КУПИТЬ' if d>THRESHOLD*100
             else '🔴 ПРОДАТЬ' if d<-THRESHOLD*100
             else '⚪ ДЕРЖАТЬ')
    return out

all_signals = {}
print("Торговые сигналы (последние 10 дней):")
print("="*70)
for t in TICKERS:
    s = get_signals(t); all_signals[t] = s
    c = s[~s['Сигнал'].str.contains('аномалия')]
    buy = (c['Сигнал'].str.contains('КУПИТЬ')).sum()
    sell= (c['Сигнал'].str.contains('ПРОДАТЬ')).sum()
    hold= (c['Сигнал'].str.contains('ДЕРЖАТЬ')).sum()
    print(f"\n{t}  |  BUY:{buy}  SELL:{sell}  HOLD:{hold}")
    print(c[['Цена_факт','Прогноз','Изменение_%','Сигнал']].tail(10).to_string())

---
## 💾 Шаг 8 — Сохранение модели

После выполнения скачайте `stocsignal_model.pkl`:
`Files (левая панель) → stocsignal_model.pkl → ПКМ → Download`

In [ ]:
import pickle, os

bundle = {
    'models':    {t: trained[t]['GBR ★'] for t in TICKERS},
    'lstm':      {t: trained[t]['LSTM']       for t in TICKERS},
    'features':  FEATURES, 'target': TARGET,
    'threshold': THRESHOLD, 'lstm_window': LSTM_WIN,
    'tickers':   TICKERS, 'ticker_start': TICKER_START,
    'saved_at':  str(date.today()),
    'last_30':   {t: feat[t].tail(30) for t in TICKERS},
}
MODEL_PATH = 'stocsignal_model.pkl'
with open(MODEL_PATH,'wb') as f: pickle.dump(bundle, f)
size = os.path.getsize(MODEL_PATH)/1024
print(f"✅ Сохранено: {MODEL_PATH}  ({size:.0f} KB)")
print(f"   Дата: {bundle['saved_at']}  |  Тикеры: {TICKERS}")
with open(MODEL_PATH,'rb') as f: ch = pickle.load(f)
print(f"✅ Проверка: {len(ch['models'])} GBR + {len(ch['lstm'])} LSTM")
print("\n📥 Скачайте файл: Files → stocsignal_model.pkl → ПКМ → Download")

---
## 🤖 Шаг 9 — Telegram-бот

Запускается прямо в Colab. Пока ячейка выполняется — бот работает.

**Перед запуском:**
1. @BotFather → /newbot → скопируйте токен в `BOT_TOKEN`
2. @userinfobot → скопируйте Your ID в `CHAT_ID`
3. Запустите ячейку — получите сигнал сразу и каждый день в 09:00 МСК

In [ ]:
# ── ВСТАВЬТЕ СЮДА СВОИ ДАННЫЕ ─────────────────────────────────────
BOT_TOKEN = "ВАШ_ТОКЕН_ОТ_BOTFATHER"
CHAT_ID   = "ВАШ_CHAT_ID"
# ───────────────────────────────────────────────────────────────────

!pip install python-telegram-bot apscheduler --quiet

import asyncio, pickle
from datetime import date as _date
import aiohttp, aiomoex, pandas as pd, numpy as np
from telegram import Bot
from apscheduler.schedulers.asyncio import AsyncIOScheduler
import nest_asyncio; nest_asyncio.apply()

with open('stocsignal_model.pkl','rb') as f: B = pickle.load(f)
MODELS = B['models']; FEATURES = B['features']
TICKERS = B['tickers']; THRESHOLD = B['threshold']
T_START = B['ticker_start']
NAMES = {'SBER':'Сбербанк','GAZP':'Газпром','LKOH':'Лукойл'}

def rsi2(s, p=14):
    d=s.diff(); g=d.clip(lower=0).rolling(p).mean()
    l=(-d.clip(upper=0)).rolling(p).mean()
    return 100-100/(1+g/l.replace(0,np.nan))

async def load_fresh(ticker):
    async with aiohttp.ClientSession() as session:
        data = await aiomoex.get_board_history(
            session, ticker, start=T_START.get(ticker,'2023-01-01'),
            end=_date.today().isoformat(),
            columns=['TRADEDATE','OPEN','HIGH','LOW','CLOSE','VOLUME'])
    df = pd.DataFrame(data)
    df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'])
    return df.set_index('TRADEDATE').sort_index().astype(float)

def build_feat2(df):
    d = df.copy()
    d['VOLUME_LOG']=np.log1p(d['VOLUME'])
    d['Lag_1']=d['CLOSE'].shift(1); d['Lag_5']=d['CLOSE'].shift(5)
    d['Lag_10']=d['CLOSE'].shift(10); d['MA_5']=d['CLOSE'].rolling(5).mean()
    d['MA_20']=d['CLOSE'].rolling(20).mean(); d['STD_10']=d['CLOSE'].rolling(10).std()
    d['RSI_14']=rsi2(d['CLOSE']); d['Return_1d']=d['CLOSE'].pct_change()
    d['DayOfWeek']=d.index.dayofweek; d['Month']=d.index.month
    return d.dropna(subset=FEATURES)

def predict_signal(ticker, df_feat):
    cur  = df_feat['CLOSE'].iloc[-1]
    pred = MODELS[ticker].predict(df_feat[FEATURES].iloc[[-1]])[0]
    d    = (pred-cur)/cur*100
    if abs(d)>50: return None
    if d>THRESHOLD*100:   e,a = '🟢','КУПИТЬ'
    elif d<-THRESHOLD*100: e,a = '🔴','ПРОДАТЬ'
    else:                   e,a = '⚪','ДЕРЖАТЬ'
    return (f"{e} *{ticker}* — {NAMES.get(ticker,ticker)}\n"
            f"Сейчас: `{cur:.2f}` руб.\n"
            f"Прогноз: `{pred:.2f}` руб.\n"
            f"Изменение: `{d:+.2f}%`\n"
            f"Сигнал: *{a}*")

async def send_signals():
    print(f"[{_date.today()}] Отправляем...")
    bot = Bot(token=BOT_TOKEN); msgs = []
    for t in TICKERS:
        try:
            df = build_feat2(await load_fresh(t))
            txt = predict_signal(t, df)
            if txt: msgs.append(txt); print(f"  {t} ✓")
        except Exception as e:
            print(f"  {t} ошибка: {e}")
    text = (f"📊 *StokSignal* | {_date.today()}\n"
            f"_Ежедневный прогноз по акциям МОЕХ_\n\n"
            + "\n\n─"*8 + "\n\n".join(msgs)
            + "\n\n⚠️ _Сигналы информационные, не инвестиционная рекомендация._"
            ) if msgs else f"📊 *StokSignal* | {_date.today()}\n\nВсе акции — HOLD."
    await bot.send_message(chat_id=CHAT_ID, text=text, parse_mode='Markdown')
    print(f"  ✅ Отправлено в Telegram")

async def run():
    await send_signals()
    sch = AsyncIOScheduler(timezone='Europe/Moscow')
    sch.add_job(send_signals, 'cron', hour=9, minute=0)
    sch.start()
    print()
    print("✅ Бот запущен!")
    print("   — Сигнал уже отправлен в Telegram")
    print("   — Следующий: завтра в 09:00 МСК")
    print("   — Пока ячейка выполняется — бот работает")
    print("   — Нажмите ■ (Stop) чтобы остановить")
    while True: await asyncio.sleep(3600)

asyncio.run(run())

---
## ✅ Итоги

| Шаг | Что сделали |
|-----|------------|
| 1 | Установили зависимости |
| 2 | Загрузили данные SBER/GAZP/LKOH с ISS MOEX |
| 3 | Очистили (ffill, IQR, log Volume) |
| 4 | Построили 11 признаков |
| 5 | Обучили 4 модели + LSTM, выбрали GBR |
| 6 | Построили 3 графика как в презентации |
| 7 | Сгенерировали сигналы BUY/SELL/HOLD |
| 8 | Сохранили модель в `stocsignal_model.pkl` |
| 9 | Запустили Telegram-бот (ежедневно 09:00 МСК) |

**Метрики GBR:** SBER ~0.49% · GAZP ~1.54% · LKOH ~0.48% — все < 2% ✅